# Element Categories of the Object Types

Assigns every `ObjectTypes` member an object category (node, connecting element, bypass element, table, table row, signal model element, ...) plus the toolkit method(s) to create it - meant to be merged into the object-types RST reference by `result_props_merge_descriptions.ipynb`, as a label under each object type's heading.

Wherever the Toolkit API itself can tell, the category is probed, not hand-maintained: in a fresh model per network type with two nodes, every object type is tried with `AddNewConnectingElement`, `AddNewBypassElement`, and `InsertElement` + `GetEndNodes`/`AddTableRow`. The remaining categories (signal model, graphical objects, settings, ...) have no API call that distinguishes them and are assigned by hand further down. The final check fails if any object type ends up without a category, so object types added in a future SIR 3S version can't slip through unlabelled.

## Import/Init

In [1]:
from sir3stoolkit.core import wrapper

In [2]:
wrapper.Initialize_Toolkit(r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2")

[2026-11-04 20:40:10,754] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-11-04 20:40:10,754] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-11-04 20:40:10,777] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-11-04 20:40:11,006] INFO in sir3stoolkit.core.wrapper: [Initialization] Sir3S_Toolkit.dll file version 90.15.11.0 (matches this sir3stoolkit version).


In [3]:
s3s = wrapper.SIR3S_Model()

[2026-11-04 20:40:11,142] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


In [4]:
import json
import logging
from pathlib import Path

## Configuration

All four network types, as in `object_types.ipynb` - besides the categories, the probe also records which network types each object type can be inserted into (`insertable_in`). The probe models are throwaway files created next to this notebook.

In [5]:
network_names = ["Water", "DistrictHeating", "Gas", "Steam"]

output_dir = (Path.cwd().resolve() / "./output/.").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

table_name_mapping = json.loads((output_dir / "object_type_table_map_independent.json").read_text(encoding="utf-8"))
object_types = list(s3s.ObjectTypes)  # all members, incl. Unknown (not reachable via ObjectTypes(0..149))
len(object_types)

149

## 1) Probe every object type per network type

Most attempts are expected to fail (a node is no connecting element, a pipe table has no end nodes, ...); each failure is logged by the toolkit as an error, so the wrapper's logger is silenced while probing. `AddNewConnectingElement`/`AddNewBypassElement` only count as accepted if they return a real tk.

In [6]:
s3s.AllowSirMessageBox(bAllow=False)  # avoid blocking popups during unattended batch insertion
s3s.EnableOrDisableOutputComments(outputComments=False)

In [7]:
def try_call(func, *args):
    try:
        return func(*args)
    except Exception:
        return None

def is_tk(tk):
    return bool(tk) and tk != "-1"

def probe_network(network_name):
    db_path = Path(f"./category_probe_{network_name}.db3").resolve()
    if db_path.exists():
        db_path.unlink()
    s3s.NewModel(dbName=str(db_path), providerType=s3s.ProviderTypes.SQLite, namedInstance="",
                 netType=getattr(s3s.NetworkType, network_name), modelDescription="Category probe",
                 userID="", password="")
    node_i = s3s.AddNewNode(tkCont="-1", name="NodeI", typ="QKON", x=0, y=0, z=0, qm_PH=0,
                            symbolFactor=1, description="", idRef="", kvr=0)
    node_k = s3s.AddNewNode(tkCont="-1", name="NodeK", typ="QKON", x=100, y=0, z=0, qm_PH=0,
                            symbolFactor=1, description="", idRef="", kvr=0)
    probe = {}
    for ot in object_types:
        record = {}
        tk = try_call(s3s.AddNewConnectingElement, "-1", node_i, node_k, 50.0, 0.0, 0.0, ot, 100.0, 1.0, 0.0, "", "")
        record["connecting"] = is_tk(tk)
        tk = try_call(s3s.AddNewBypassElement, "-1", node_i, 0.0, 50.0, 0.0, 1.0, ot, "", "")
        record["bypass"] = is_tk(tk)
        tk = try_call(s3s.InsertElement, ot, "-1")
        record["insertable"] = is_tk(tk)
        if record["insertable"]:
            # GetEndNodes returns fkKI, fkKK, fkKI2, fkKK2 ('-1' if unconnected) - or nothing for types without end nodes
            record["has_end_nodes"] = any(try_call(s3s.GetEndNodes, tk) or [])
            row = try_call(s3s.AddTableRow, tk)
            if row and is_tk(row[0]):
                record["row_type"] = row[1].name
        probe[ot.name] = record
    s3s.CloseModel(False)
    return probe

In [8]:
toolkit_logger = logging.getLogger("sir3stoolkit.core.wrapper")  # sets its own level, so not via the parent
previous_level = toolkit_logger.level
toolkit_logger.setLevel(logging.CRITICAL)
try:
    probes = {network_name: probe_network(network_name) for network_name in network_names}
finally:
    toolkit_logger.setLevel(previous_level)

{network_name: sum(r["insertable"] for r in probe.values()) for network_name, probe in probes.items()}

{'Water': 141, 'DistrictHeating': 140, 'Gas': 129, 'Steam': 127}

## 1b) Probe the `SIR3S_View` insert methods

`SIR3S_View` has its own insert methods for some object types (houses, customers, streets, net valves, hydrants, texts, numerical displays, directional arrows) - each called once per network type here, and `GetObjectTypeof_Key` on the returned tk tells which object type it created. Inside SirGraf, `SIR3S_Model` and `SIR3S_View` share one toolkit (`StaticUI.Toolkit`); outside, each creates its own, so the view is given the model's toolkit to see the models opened here.

The `AddExternal...` methods (polylines, polygons, texts, arrows, rectangles, ellipses) are probed too, but create no model element: their tk's object type is `Unknown` and no object type's element count changes - they're view overlays, not the `POLY`/`PLYG`/`GTXT`/`ARRW`/`RECT`/`CIRC` elements.

In [9]:
view = wrapper.SIR3S_View()
view.toolkit = s3s.toolkit
view.EnableOrDisableOutputComments(outputComments=False)

[2026-11-04 20:40:37,481] INFO in sir3stoolkit.core.wrapper: Initialization complete


In [10]:
def probe_view_methods(network_name):
    db_path = Path(f"./category_probe_view_{network_name}.db3").resolve()
    if db_path.exists():
        db_path.unlink()
    s3s.NewModel(dbName=str(db_path), providerType=s3s.ProviderTypes.SQLite, namedInstance="",
                 netType=getattr(s3s.NetworkType, network_name), modelDescription="Category probe (view)",
                 userID="", password="")
    cont = s3s.GetMainContainer()[0]
    node_i = s3s.AddNewNode(tkCont="-1", name="NodeI", typ="QKON", x=0, y=0, z=0, qm_PH=0,
                            symbolFactor=1, description="", idRef="", kvr=0)
    node_k = s3s.AddNewNode(tkCont="-1", name="NodeK", typ="QKON", x=100, y=0, z=0, qm_PH=0,
                            symbolFactor=1, description="", idRef="", kvr=0)
    pipe = s3s.AddNewPipe("-1", node_i, node_k, 100.0, "LINESTRING (0 0, 100 0)", "", "100", 0.1, "", "", 0)
    dh_consumer = ""
    if network_name == "DistrictHeating":
        dh_consumer = s3s.AddNewConnectingElement("-1", node_i, node_k, 50.0, 20.0, 0.0,
                                                  s3s.ObjectTypes.DistrictHeatingConsumer, 100.0, 1.0, 0.0, "", "")

    created = {}
    def record(method_name, *args):
        tk = try_call(getattr(view, method_name), *args)
        if is_tk(tk):
            created[method_name] = s3s.GetObjectTypeof_Key(tk).name
        return tk

    street = record("AddNewStreet", "Street", "1", "Place", "District", "")
    house = record("AddNewHouse", 10.0, 10.0, 1.0, street, 1, "", 12345, "",
                   "" if dh_consumer else node_i, dh_consumer, "")
    division_type = {"Water": "W-", "DistrictHeating": "F-", "Gas": "G-"}.get(network_name, "")
    record("AddNewCustomer", 12.0, 12.0, 0.0, 1.0, house, 100.0, "", "", "", division_type, "", "")
    record("AddNewValveOnPipe", pipe, view.NetValveTypes.GATE_VALVE, -2.0, "", "",
           view.NetValvePostures.STATIC_OPEN_CLOSE, "", True, "")
    record("AddNewHydrant", 20.0, 20.0, 0.0, view.Hydrant_Type.SUBSURFACE, 1.0, node_i, 10.0, 100.0, 0.1,
           1.5, 2.0, view.Hydrant_QM_SOLL.QM_SOLL_0, view.Hydrant_Activity.INACTIVE, "", "", "")
    record("AddNewText", cont, 30.0, 30.0, 0, "Text", 0.0, "Arial", 10.0, False, False, False, "", "")
    record("AddNewNumericalDisplay", cont, 40.0, 40.0, 0, 0.0, "Arial", 10.0, False, False, False, "",
           False, node_i, "Zkor", "", "", 2, False)
    record("AddNewDirectionalArrow", cont, 50.0, 50.0, 0, 0.5, 0, True, 1.0, "", pipe, "QMAV", 0.01)
    record("AddExternalPolyline", [0.0, 10.0], [0.0, 10.0], 0, 0.5, False, cont)
    record("AddExternalPolygon", [0.0, 10.0, 10.0], [0.0, 0.0, 10.0], 0, 0, 0.5, True, cont)
    record("AddExternalText", 60.0, 60.0, 0, "Text", 0.0, 10.0, False, False, False, cont)
    record("AddExternalArrow", 70.0, 70.0, 0, 0, 0.5, True, 1.0, cont)
    record("AddExternalRectangle", 0.0, 10.0, 10.0, 0.0, 0, 0, 0.5, False, False, cont)
    record("AddExternalEllipse", 0.0, 10.0, 10.0, 0.0, 0, 0, 0.5, False, cont)
    s3s.CloseModel(False)
    return created

In [11]:
toolkit_logger.setLevel(logging.CRITICAL)
try:
    view_probes = {network_name: probe_view_methods(network_name) for network_name in network_names}
finally:
    toolkit_logger.setLevel(previous_level)

view_created = {}
for created in view_probes.values():
    for method_name, object_type_name in created.items():
        view_created.setdefault(method_name, set()).add(object_type_name)
view_created

{'AddNewStreet': {'Street'},
 'AddNewHouse': {'House'},
 'AddNewCustomer': {'Demand'},
 'AddNewValveOnPipe': {'NetValve'},
 'AddNewHydrant': {'Hydrant'},
 'AddNewText': {'Text'},
 'AddNewNumericalDisplay': {'NumericalDisplay'},
 'AddNewDirectionalArrow': {'DirectionalArrow'},
 'AddExternalPolyline': {'Unknown'},
 'AddExternalPolygon': {'Unknown'},
 'AddExternalText': {'Unknown'},
 'AddExternalArrow': {'Unknown'},
 'AddExternalRectangle': {'Unknown'},
 'AddExternalEllipse': {'Unknown'}}

In [12]:
# AddNew... must each create one (real) object type in at least one network type (AddNewHydrant: Water only),
# AddExternal... none.
view_method_by_type = {}
for method_name in [m for m in dir(view) if m.startswith(("AddNew", "AddExternal")) and "_using_" not in m and not m.endswith("Point")]:
    object_type_names = view_created.get(method_name, set())
    if method_name.startswith("AddExternal"):
        assert object_type_names <= {"Unknown"}, f"{method_name} created {object_type_names}"
        continue
    assert len(object_type_names) == 1 and "Unknown" not in object_type_names, f"{method_name}: {object_type_names}"
    view_method_by_type[object_type_names.pop()] = method_name
view_method_by_type

{'Demand': 'AddNewCustomer',
 'DirectionalArrow': 'AddNewDirectionalArrow',
 'House': 'AddNewHouse',
 'Hydrant': 'AddNewHydrant',
 'NumericalDisplay': 'AddNewNumericalDisplay',
 'Street': 'AddNewStreet',
 'Text': 'AddNewText',
 'NetValve': 'AddNewValveOnPipe'}

## 2) Categories derived from the probe

Combined over all network types (e.g. `Compressor` is only accepted in Gas, `DistrictHeatingConsumer` only in DistrictHeating - the category is the same either way).

- `Node`: the only object type created with `AddNewNode`.
- `Pipe` is rejected by `AddNewConnectingElement`, but has end nodes per `GetEndNodes` - it has its own method, `AddNewPipe`.
- `HeatExchanger` is rejected by `AddNewConnectingElement` ("Heat Exchanger (FWWU) not supported by this Method"), but has end nodes per `GetEndNodes` - four of them (fkKI, fkKK, fkKI2, fkKK2), per its docstring.
- A table is any object type `AddTableRow` accepts, which also returns its row type.

In [13]:
def any_network(object_type_name, key):
    return any(probes[n][object_type_name].get(key) for n in network_names)

def has_end_nodes(object_type_name):
    return any_network(object_type_name, "has_end_nodes")

categories = {}
for ot in object_types:
    name = ot.name
    entry = {"table_name": table_name_mapping.get(name), "source": "toolkit"}
    if name == "Node":
        entry.update(category="node", create_with=["SIR3S_Model.AddNewNode"])
    elif any_network(name, "connecting"):
        entry.update(category="connecting_element", create_with=["SIR3S_Model.AddNewConnectingElement"])
    elif name == "Pipe" and has_end_nodes(name):
        entry.update(category="connecting_element", create_with=["SIR3S_Model.AddNewPipe"])
    elif any_network(name, "bypass"):
        entry.update(category="bypass_element", create_with=["SIR3S_Model.AddNewBypassElement"])
    elif name == "HeatExchanger" and has_end_nodes(name):
        entry.update(category="four_end_node_element", create_with=["SIR3S_Model.InsertElement"])
    else:
        row_types = {probes[n][name]["row_type"] for n in network_names if "row_type" in probes[n][name]}
        if not row_types:
            continue
        assert len(row_types) == 1, f"{name}: different row types per network type: {row_types}"
        row_type = row_types.pop()
        entry.update(category="table", create_with=["SIR3S_Model.InsertElement"], row_type=row_type)
        categories[row_type] = {
            "table_name": table_name_mapping.get(row_type), "source": "toolkit",
            "category": "table_row", "create_with": ["SIR3S_Model.AddTableRow"], "table_type": name,
        }
    categories[name] = entry

# Sanity check: everything with end nodes got a node-based category.
for ot in object_types:
    if has_end_nodes(ot.name):
        assert categories.get(ot.name, {}).get("category") in ("connecting_element", "bypass_element", "four_end_node_element"), ot.name

from collections import Counter
Counter(c["category"] for c in categories.values())

Counter({'table_row': 27,
         'table': 27,
         'connecting_element': 15,
         'bypass_element': 5,
         'node': 1,
         'four_end_node_element': 1})

## 3) Manual categories

Everything the probe can't tell apart, assigned by hand by SIR 3S table code (object type name for the three types without a table code). No `create_with` here: `InsertElement` returns a tk for most of them, but that doesn't show the result is a usable element (e.g. a pipe vertex without its pipe) - only methods shown to create a real instance are linked (steps 2 and 4).

- `signal_model_element`: the signal model's blocks (cf. PT3S's `vRXXX_nodes`, plus `RSTE` and the transition symbol `RUES`); `signal_model_connection`: `CRGL`, the line connecting them (`fkKi`/`fkKk`).
- `graphical_object`: drawing/annotation objects without hydraulic meaning.
- `container_or_block`: containers, blocks and the nodes connecting them.
- `attached_element`: belongs to one node (`HAUS`), pipe (`NSCH`), node+pipe (`HYDR`) or house (`VERB`).
- `sub_record`: child or link record of other elements (`Fk`/two `fk...` properties).
- `settings`: model-wide settings.
- `reference_data`: records other object types reference via an `fk<table code>` property (e.g. `FSTF` via `KNOT.Fkfstf`, `SOKO` via `FWEA.Fksoko`) and that have no results of their own.
- `not_insertable`: no table in SIR 3S behind the enum member ("Could not find Repository").
- `unknown`: not sure - deliberately not guessed.

In [14]:
MANUAL_CATEGORIES = {
    "signal_model_element": ["RADD", "RDIV", "RFKT", "RHYS", "RINT", "RLSR", "RLVG", "RMES", "RMMA",
                             "RMUL", "RPID", "RPT1", "RSLW", "RSTE", "RSTN", "RTOT", "RUES"],
    "signal_model_connection": ["CRGL"],
    "graphical_object": ["ARRW", "CIRC", "GTXT", "NRCV", "OVAL", "PLYG", "POLY", "RECT", "RPFL", "RRCT"],
    "container_or_block": ["BREF", "CONT", "VKNO"],
    "attached_element": ["HAUS", "HYDR", "NSCH", "VERB"],
    "sub_record": ["DPGR_DPKT", "FQPS_BZ", "PGRP_PUMP", "RMES_DPTS", "ROHR_VRTX"],
    "settings": ["ALLG", "ATMO", "GRAV", "PARZ", "REGP", "SIRGRAF"],
    "reference_data": ["DPGR", "DPKT", "FQPS", "FSTF", "GKMP", "GMIX", "LTGR", "PZON", "SOKO", "STRASSE",
                       "UTMP"],
    "not_insertable": ["RART_ControlMode", "Unknown"],
    "unknown": ["AGSN", "EBES", "ELEMENTQUERY", "FKNL", "FWBZ", "LAYR", "PARI", "PGRP", "RART", "SPLZ",
                "TRVA", "USCH", "VRCT", "WBLZ"],
}

name_by_table_name = {table_name: name for name, table_name in table_name_mapping.items()}
known_names = {ot.name for ot in object_types}

manual_added = 0
for category, keys in MANUAL_CATEGORIES.items():
    for key in keys:
        name = name_by_table_name.get(key, key)
        assert name in known_names, f"{key}: no such object type"
        assert name not in categories, f"{key}: already categorized as {categories[name]['category']} by the probe"
        categories[name] = {
            "table_name": table_name_mapping.get(name), "source": "manual", "category": category,
            "create_with": [],
        }
        manual_added += 1

missing = sorted(ot.name for ot in object_types if ot.name not in categories)
assert not missing, f"Object types without a category: {missing}"
print(f"{manual_added} manual entries, {len(categories)} object types categorized in total")

73 manual entries, 149 object types categorized in total


## 4) Prefer the dedicated `SIR3S_View` methods

For object types a `SIR3S_View` method was shown to create (step 1b), that method becomes `create_with` - it sets up the references (street, node, house, pipe, observed element) in the same call. The category itself stays as assigned above.

In [15]:
for object_type_name, method_name in view_method_by_type.items():
    categories[object_type_name]["create_with"] = [f"SIR3S_View.{method_name}"]
{name: (categories[name]["table_name"], categories[name]["category"]) for name in view_method_by_type}

{'Demand': ('VERB', 'attached_element'),
 'DirectionalArrow': ('RPFL', 'graphical_object'),
 'House': ('HAUS', 'attached_element'),
 'Hydrant': ('HYDR', 'attached_element'),
 'NumericalDisplay': ('NRCV', 'graphical_object'),
 'Street': ('STRASSE', 'reference_data'),
 'Text': ('GTXT', 'graphical_object'),
 'NetValve': ('NSCH', 'attached_element')}

## 5) Network types

`insertable_in`: the network types whose probe model `InsertElement` accepted the object type in (it rejects the others with e.g. "Element Type cannot be inserted into this Network Type"). Table rows only exist within their table, so they get their table's network types.

In [16]:
for name, entry in categories.items():
    source_name = entry.get("table_type", name)
    entry["insertable_in"] = [n for n in network_names if probes[n][source_name]["insertable"]]
{name: categories[name]["insertable_in"] for name in ["Hydrant", "Compressor", "DistrictHeatingConsumer", "PipeTable_Row", "RART_ControlMode"]}

{'Hydrant': ['Water'],
 'Compressor': ['Gas'],
 'DistrictHeatingConsumer': ['DistrictHeating'],
 'PipeTable_Row': ['Water', 'DistrictHeating', 'Gas', 'Steam'],
 'RART_ControlMode': []}

## 6) Save

In [17]:
out_path = output_dir / "object_type_categories.json"
out_path.write_text(json.dumps(dict(sorted(categories.items())), indent=2), encoding="utf-8")
print(f"Wrote {out_path}")

Wrote C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_type_categories.json


In [18]:
Counter(c["category"] for c in categories.values())

Counter({'table_row': 27,
         'table': 27,
         'signal_model_element': 17,
         'connecting_element': 15,
         'unknown': 14,
         'reference_data': 11,
         'graphical_object': 10,
         'settings': 6,
         'bypass_element': 5,
         'sub_record': 5,
         'attached_element': 4,
         'container_or_block': 3,
         'not_insertable': 2,
         'node': 1,
         'four_end_node_element': 1,
         'signal_model_connection': 1})